# 05 — Group-Stage Analysis

**Main question:** Did the passing-network profile that separates QF+ teams from eliminated teams already appear during the group stage, or was it mainly driven by knockout-stage matches?

**Why this matters:** In knockout rounds only stronger teams remain, which inflates any metric comparison. Restricting to the group stage gives every team three matches on equal footing — a cleaner signal.

**Input:** `data/processed/team_match_network_features_normalized.csv`  
**Output:** `outputs/tables/group_stage_qf_comparison.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROC_DIR   = Path('../data/processed')
FIG_DIR    = Path('../outputs/figures')
TABLE_DIR  = Path('../outputs/tables')
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROC_DIR / 'team_match_network_features_normalized.csv')
print(f'Full dataset: {len(df)} rows')

## 1. Inspect the stage column

In [ ]:
print('Unique stage values:')
print(df['stage'].value_counts().to_string())

## 2. Filter to group stage

Match any stage label containing "group" (case-insensitive) so the filter works even if the label changes.

In [ ]:
gs = df[df['stage'].str.contains('group', case=False, na=False)].copy()

print('── Validation ───────────────────────────────────────────────')
print(f'  Total rows (all stages)      : {len(df)}')
print(f'  Group-stage rows             : {len(gs)}')
print(f'  Unique matches in group stage: {gs["match_id"].nunique()}')
print(f'  Unique teams in group stage  : {gs["team"].nunique()}')
print()
print(f'  reached_qf = 1 (QF+ teams)  : {(gs["reached_qf"] == 1).sum()} rows  '
      f'({gs[gs["reached_qf"]==1]["team"].nunique()} teams × 3 matches)')
print(f'  reached_qf = 0 (non-QF)     : {(gs["reached_qf"] == 0).sum()} rows  '
      f'({gs[gs["reached_qf"]==0]["team"].nunique()} teams × 3 matches)')
print('─────────────────────────────────────────────────────────────')

## 3. QF+ vs non-QF comparison within the group stage

In [ ]:
COMPARE_COLS = [
    'completed_passes',
    'unique_passing_pairs',
    'network_density',
    'top_player_reliance',
    'final_third_entries',
    'final_third_entries_per_100_passes',
    'connections_per_100_passes',
    'passes_per_connection',
]

qf_gs  = gs[gs['reached_qf'] == 1]
non_gs = gs[gs['reached_qf'] == 0]

rows = []
for col in COMPARE_COLS:
    qf_mean  = qf_gs[col].mean()
    non_mean = non_gs[col].mean()
    qf_med   = qf_gs[col].median()
    non_med  = non_gs[col].median()
    mean_diff = qf_mean - non_mean
    mean_pct  = mean_diff / non_mean * 100 if non_mean else np.nan
    rows.append({
        'metric':              col,
        'qf_mean':             round(qf_mean,  3),
        'non_qf_mean':         round(non_mean, 3),
        'mean_difference':     round(mean_diff, 3),
        'mean_pct_difference': round(mean_pct,  1),
        'qf_median':           round(qf_med,   3),
        'non_qf_median':       round(non_med,  3),
        'median_difference':   round(qf_med - non_med, 3),
    })

comparison = pd.DataFrame(rows)

# Save
comparison.to_csv(TABLE_DIR / 'group_stage_qf_comparison.csv', index=False)
print('Saved: outputs/tables/group_stage_qf_comparison.csv')
print()
comparison

In [ ]:
# Readable printout — compare group-stage to full-dataset findings side-by-side
# Full-dataset values computed from the whole df for reference
print('=' * 86)
print(f'  {"Metric":<38}  {"Group QF+":>9}  {"Group Non-QF":>12}  {"Diff":>7}  {"Direction"}')
print('=' * 86)
for _, r in comparison.iterrows():
    arrow = '▲' if r['mean_pct_difference'] > 0 else '▼'
    print(f'  {r["metric"]:<38}  {r["qf_mean"]:>9.3f}  {r["non_qf_mean"]:>12.3f}  '
          f'{arrow}{abs(r["mean_pct_difference"]):>4.1f}%')
print('=' * 86)

## 4. Visualizations

In [ ]:
QF_COLOR  = '#1E88E5'
ELM_COLOR = '#E53935'

def boxplot(ax, col, title, ylabel=None):
    """Single styled boxplot comparing non-QF vs QF+ for a given column."""
    bp = ax.boxplot(
        [non_gs[col].dropna(), qf_gs[col].dropna()],
        labels=['Non-QF', 'QF+'],
        patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
        flierprops=dict(marker='o', markersize=4, alpha=0.4),
    )
    for patch, color in zip(bp['boxes'], [ELM_COLOR, QF_COLOR]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    # Annotate medians
    for i, vals in enumerate([non_gs[col], qf_gs[col]]):
        med = vals.dropna().median()
        ax.text(i + 1, med, f' {med:.2f}', va='center', fontsize=8.5, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=8.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)


# ── Four boxplots in a 2×2 grid ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(
    'Group Stage Only — QF+ vs Eliminated Teams\nFIFA World Cup 2022 (96 team-match rows)',
    fontsize=13, fontweight='bold', y=1.01,
)

boxplot(axes[0, 0], 'completed_passes',
        'Completed Passes', 'Total completed passes')

boxplot(axes[0, 1], 'final_third_entries_per_100_passes',
        'Final-Third Entries per 100 Passes', 'Entries per 100 passes')

boxplot(axes[1, 0], 'connections_per_100_passes',
        'Connections per 100 Passes', 'Unique connections per 100 passes')

boxplot(axes[1, 1], 'passes_per_connection',
        'Passes per Connection', 'Avg passes per unique pair')

plt.tight_layout()
plt.savefig(FIG_DIR / '05_group_stage_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/05_group_stage_boxplots.png')

In [ ]:
# ── Scatter: passes vs final-third entries (group stage only) ─────────────
fig, ax = plt.subplots(figsize=(9, 6))

for val, label, color in [(0, 'Non-QF', ELM_COLOR), (1, 'QF+', QF_COLOR)]:
    sub = gs[gs['reached_qf'] == val]
    ax.scatter(
        sub['completed_passes'], sub['final_third_entries'],
        c=color, alpha=0.55, s=50, label=label,
        edgecolors='white', linewidths=0.3,
    )
    # Trend line
    sub_clean = sub.dropna(subset=['completed_passes', 'final_third_entries'])
    z = np.polyfit(sub_clean['completed_passes'], sub_clean['final_third_entries'], 1)
    x_line = np.linspace(sub_clean['completed_passes'].min(),
                         sub_clean['completed_passes'].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line),
            color=color, linewidth=1.8, linestyle='--', alpha=0.85)

    # Group mean as a large marker
    ax.scatter(
        sub['completed_passes'].mean(), sub['final_third_entries'].mean(),
        c=color, s=180, marker='D', edgecolors='black', linewidths=1.2,
        zorder=5, label=f'{label} mean',
    )

ax.set_xlabel('Completed passes', fontsize=11)
ax.set_ylabel('Final-third entries (raw)', fontsize=11)
ax.set_title(
    'Group Stage: Completed Passes vs Final-Third Entries\n'
    'Diamonds = group means  |  Dashed lines = trend per group',
    fontsize=11, fontweight='bold',
)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIG_DIR / '05_group_stage_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/05_group_stage_scatter.png')

## 5. Interpretation

### Did QF+ teams still complete more passes during the group stage?

Yes. The volume gap is already present in the group stage. QF+ teams averaged more completed passes per match than non-QF teams even before knockout rounds. The separation is visible in both the mean and median, and the scatter plot shows the QF+ mean diamond sitting noticeably to the right.

This suggests possession dominance is not a knockout-stage artifact — it was characteristic of these teams from matchday 1.

### Did QF+ teams still have more raw final-third entries?

Yes. QF+ teams generated more total final-third entries during the group stage. The scatter plot shows both a rightward (more passes) and upward (more entries) shift for QF+ teams relative to non-QF teams.

### Did QF+ teams have lower or higher final-third entries per 100 passes?

Lower — consistent with the full-dataset finding. When controlling for passing volume, non-QF teams converted a slightly higher share of their passes into final-third entries. This pattern holds in the group stage, not just overall.

This is an important nuance: QF+ teams' raw entry advantage was largely a volume effect. Their individual passes were not more likely to reach the final third.

### Did QF+ teams show more repeated passing relationships (higher passes per connection)?

Yes. The passes per connection gap is present in the group stage. QF+ teams used each passing connection more frequently, pointing to a more structured, repetitive passing style rather than a more varied one. Fewer unique connections per 100 passes, but each connection used more often — a tighter, more habitual network.

### Does the group-stage analysis support or weaken the earlier overall finding?

It **supports** it. The patterns identified across all stages — more passes, denser networks, higher passes per connection, slightly lower rate-based final-third penetration — are already visible in group-stage matches alone. They were not created by the selective survival of stronger teams in later rounds.

---

**Caution:** These patterns come from a single tournament (n = 32 teams, 3 matches each). Multiple rows per team means the rows are not independent — the same team's playing style appears in all three of its group matches. Treat these as descriptive patterns, not statistically confirmed predictors.

> **Working conclusion:** QF+ teams' passing-network profile — higher volume, denser connectivity, more repeated use of the same connections — was already present in the group stage. This suggests these metrics capture something real about how these teams play, not just how far they happened to go. The next step would be to look at per-team averages (one row per team, not one per match) to remove the repeated-rows issue and compare team-level profiles directly.

The group-stage-only results strengthen the overall finding that quarterfinal teams had a distinct passing-network profile from the beginning of the tournament. QF+ teams completed substantially more passes, had slightly denser networks, and generated more raw final-third entries even before knockout rounds began. However, their final-third entries per 100 passes were lower, suggesting that their attacking advantage came from sustained possession volume rather than greater directness per pass. Their lower connections per 100 passes and higher passes per connection also suggest more repeated and stable passing relationships. Overall, QF+ teams appeared to advance with controlled possession, consistent passing structures, and attacking accumulation rather than pure efficiency or reduced hub reliance.